## Step 1: Install Required Libraries

In [ ]:
!pip install playwright
!playwright install chromium
!playwright install-deps

## Step 2: Import Libraries

In [ ]:
from playwright.async_api import async_playwright
import os
import re
import asyncio
from datetime import datetime
import zipfile
import base64
from IPython.display import HTML, display

print("✅ Libraries imported successfully!")

## Step 3: Configure Website & Login Details

⚡ **EDIT THIS CONFIG - ONLY PLACE TO CHANGE!**

In [ ]:
CONFIG = {
    # Website login URL
    "login_url": "https://rms-ui-dev-slp.supremelifeplatform.com/#/admin/login",
    
    # Login credentials
    "username": "ashish.admin",
    "password": "Staging123$",
    
    # Login form selectors (CSS selectors for username, password, submit button)
    # Leave empty "" to auto-detect common patterns
    "username_selector": "input[type='text'], input[name='username'], input[id='username']",
    "password_selector": "input[type='password'], input[name='password'], input[id='password']",
    "submit_selector": "button[type='submit'], input[type='submit'], button:has-text('Login')",
    
    # ==========================================
    # 🔥 AUTO-DISCOVER MODE (for 192+ pages!)
    # ==========================================
    # Set "auto_discover_links" to True to automatically find ALL links on the website
    # The script will crawl the site after login and screenshot every unique page
    
    "auto_discover_links": True,  # True = auto-find all links, False = use manual list below
    "max_pages_to_screenshot": 200,  # Maximum number of pages to screenshot (safety limit)
    "link_patterns_to_include": [    # Only screenshot URLs matching these patterns
        "/admin/",
        "/business",
        "/onboarding",
        "/#/",  # Include hash-based routes
    ],
    "link_patterns_to_exclude": [    # Skip URLs matching these patterns
        "/logout",
        "/login",
        "javascript:",
        "mailto:",
        "#",  # Skip anchor-only links
    ],
    
    # Manual pages (only used if auto_discover_links = False)
    "pages_to_screenshot": [
        "https://rms-ui-dev-slp.supremelifeplatform.com/#/admin/users",
        "https://rms-ui-dev-slp.supremelifeplatform.com/#/business",
        "https://rms-ui-dev-slp.supremelifeplatform.com/#/onboarding-sla",
    ],
    
    # Wait time between actions (seconds)
    "wait_after_login": 3,      # Wait after login completes
    "wait_before_screenshot": 2, # Wait before taking each screenshot
    "wait_between_pages": 1,     # Wait between navigating to pages (reduced for speed)
    
    # Screenshot settings
    "full_page": True,  # Capture full scrollable page (True) or just viewport (False)
    "output_folder": "screenshots",
    "zip_filename": "website_screenshots.zip",
    
    # Browser settings
    "headless": True,  # Set to False to see browser window (requires display/X server)
}

print("📋 Configuration loaded:")
print(f"  🌐 Login URL: {CONFIG['login_url']}")
print(f"  👤 Username: {CONFIG['username']}")
print(f"  🔒 Password: {'*' * len(CONFIG['password'])}")
print(f"  📸 Pages to screenshot: {len(CONFIG['pages_to_screenshot']) + 1} (login page + additional)")
print(f"  📁 Output: {CONFIG['zip_filename']}")
print("\n✅ Ready to start!")

## Step 4: Run Screenshot Automation

This will:
1. Open browser and navigate to login page
2. Fill credentials and login
3. Take screenshots of all specified pages
4. Create ZIP file with all screenshots
5. Provide download link

In [ ]:
import asyncio

def sanitize_filename(url):
    """Convert URL to safe filename."""
    # Remove protocol and special characters
    name = re.sub(r'https?://', '', url)
    name = re.sub(r'[^a-zA-Z0-9_-]', '_', name)
    # Limit length
    if len(name) > 100:
        name = name[:100]
    return name

async def take_screenshots_async(config):
    """Main function to automate login and take screenshots."""
    
    # Create output folder
    os.makedirs(config['output_folder'], exist_ok=True)
    
    screenshots_taken = []
    
    print("\n" + "="*70)
    print("🚀 Starting Screenshot Automation...")
    print("="*70 + "\n")
    
    from playwright.async_api import async_playwright
    
    async with async_playwright() as p:
        # Launch browser
        print("🌐 Launching browser...")
        browser = await p.chromium.launch(headless=config['headless'])
        context = await browser.new_context(
            viewport={'width': 1920, 'height': 1080}
        )
        page = await context.new_page()
        
        try:
            # Navigate to login page
            print(f"📍 Navigating to: {config['login_url']}")
            await page.goto(config['login_url'], wait_until='domcontentloaded', timeout=60000)
            await asyncio.sleep(3)  # Wait for dynamic content to load
            
            # Take debug screenshot before login
            debug_path = os.path.join(config['output_folder'], 'debug_before_login.png')
            await page.screenshot(path=debug_path, full_page=False)
            print(f"📸 Debug screenshot saved: {debug_path}")
            
            # Fill login form with better error handling
            print("\n🔐 Logging in...")
            
            # Find and fill username - try multiple approaches
            print("  🔍 Looking for username field...")
            try:
                # Wait for any input field to be visible
                await page.wait_for_selector('input', timeout=10000)
                
                # Try to find username field with various selectors
                username_selectors = [
                    'input[name="username"]',
                    'input[id="username"]',
                    'input[placeholder*="username" i]',
                    'input[placeholder*="user" i]',
                    'input[type="text"]',
                    'input[type="email"]',
                ]
                
                username_field = None
                for selector in username_selectors:
                    if await page.locator(selector).count() > 0:
                        username_field = page.locator(selector).first
                        print(f"  ✓ Found username field with: {selector}")
                        break
                
                if username_field:
                    await username_field.click()
                    await username_field.fill(config['username'])
                    print(f"  ✓ Username entered: {config['username']}")
                else:
                    raise Exception("Could not find username field with any selector")
                    
            except Exception as e:
                print(f"  ❌ Username field error: {e}")
                await page.screenshot(path=os.path.join(config['output_folder'], 'error_username.png'))
                raise
            
            # Find and fill password with multiple attempts
            print("  🔍 Looking for password field...")
            await asyncio.sleep(1)
            
            try:
                password_selectors = [
                    'input[name="password"]',
                    'input[id="password"]',
                    'input[type="password"]',
                    'input[placeholder*="password" i]',
                ]
                
                password_field = None
                for selector in password_selectors:
                    if await page.locator(selector).count() > 0:
                        password_field = page.locator(selector).first
                        print(f"  ✓ Found password field with: {selector}")
                        break
                
                if password_field:
                    await password_field.click()
                    await password_field.fill(config['password'])
                    print(f"  ✓ Password entered: {'*' * len(config['password'])}")
                else:
                    # Last resort: try to find all inputs and use the second one
                    all_inputs = page.locator('input')
                    input_count = await all_inputs.count()
                    if input_count >= 2:
                        password_field = all_inputs.nth(1)
                        await password_field.click()
                        await password_field.fill(config['password'])
                        print(f"  ✓ Password entered to second input field")
                    else:
                        raise Exception("Could not find password field with any selector")
                    
            except Exception as e:
                print(f"  ❌ Password field error: {e}")
                await page.screenshot(path=os.path.join(config['output_folder'], 'error_password.png'))
                print("\n  📋 Available inputs on page:")
                inputs = await page.locator('input').all()
                for i, inp in enumerate(inputs):
                    inp_type = await inp.get_attribute('type') or 'text'
                    inp_name = await inp.get_attribute('name') or 'no-name'
                    inp_id = await inp.get_attribute('id') or 'no-id'
                    print(f"    Input {i+1}: type={inp_type}, name={inp_name}, id={inp_id}")
                raise
            
            # Click submit button
            print("  🔍 Looking for submit button...")
            await asyncio.sleep(1)
            
            try:
                submit_selectors = [
                    'button[type="submit"]',
                    'input[type="submit"]',
                    'button:has-text("Login")',
                    'button:has-text("Sign in")',
                    'button:has-text("Log in")',
                    'button',  # Last resort: first button
                ]
                
                submit_button = None
                for selector in submit_selectors:
                    if await page.locator(selector).count() > 0:
                        submit_button = page.locator(selector).first
                        print(f"  ✓ Found submit button with: {selector}")
                        break
                
                if submit_button:
                    await submit_button.click()
                    print("  ✓ Login button clicked")
                else:
                    # Try pressing Enter on password field
                    await password_field.press('Enter')
                    print("  ✓ Pressed Enter on password field")
                    
            except Exception as e:
                print(f"  ⚠️  Submit button warning: {e}")
                print("  → Trying to press Enter instead...")
                await page.keyboard.press('Enter')
            
            # Wait for navigation after login
            print(f"\n⏳ Waiting {config['wait_after_login']} seconds for login to complete...")
            await asyncio.sleep(config['wait_after_login'])
            
            current_url = page.url
            print(f"✅ Login successful! Current URL: {current_url}")
            
            # Determine which pages to screenshot
            pages_to_capture = []
            
            if config.get('auto_discover_links', False):
                print(f"\n🔍 AUTO-DISCOVER MODE: Finding all links on the website...")
                
                # Get all links from the current page
                await asyncio.sleep(2)  # Wait for page to fully load
                
                # Extract all links
                all_links = await page.evaluate('''() => {
                    const links = Array.from(document.querySelectorAll('a[href]'));
                    return links.map(a => a.href).filter(href => href && href !== '');
                }''')
                
                print(f"  📊 Found {len(all_links)} total links")
                
                # Filter links
                base_domain = page.url.split('/')[2]  # Get domain
                unique_links = set()
                
                for link in all_links:
                    # Skip if link doesn't match include patterns
                    if config.get('link_patterns_to_include'):
                        if not any(pattern in link for pattern in config['link_patterns_to_include']):
                            continue
                    
                    # Skip if link matches exclude patterns
                    if config.get('link_patterns_to_exclude'):
                        if any(pattern in link for pattern in config['link_patterns_to_exclude']):
                            continue
                    
                    # Only keep links from same domain
                    if base_domain in link:
                        unique_links.add(link)
                
                pages_to_capture = list(unique_links)[:config.get('max_pages_to_screenshot', 200)]
                
                print(f"  ✅ Filtered to {len(pages_to_capture)} unique pages to screenshot")
                print(f"  ⏱️  Estimated time: ~{len(pages_to_capture) * config['wait_between_pages']} seconds")
                
            else:
                # Use manual list
                pages_to_capture = config.get('pages_to_screenshot', [])
                print(f"\n📋 Manual mode: {len(pages_to_capture)} pages specified")
            
            # Take screenshot of post-login page first
            print(f"\n📸 Taking screenshot of post-login page...")
            await asyncio.sleep(config['wait_before_screenshot'])
            
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"{sanitize_filename(current_url)}_{timestamp}.png"
            filepath = os.path.join(config['output_folder'], filename)
            
            await page.screenshot(path=filepath, full_page=config['full_page'])
            screenshots_taken.append(filename)
            print(f"  ✓ Saved: {filename}")
            
            # Screenshot all discovered/specified pages
            if pages_to_capture:
                print(f"\n📸 Taking screenshots of {len(pages_to_capture)} pages...")
                print(f"{'='*70}\n")
                
                for idx, url in enumerate(pages_to_capture, 1):
                    print(f"  [{idx}/{len(pages_to_capture)}] {url}")
                    try:
                        await page.goto(url, wait_until='domcontentloaded', timeout=30000)
                        await asyncio.sleep(config['wait_before_screenshot'])
                        
                        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                        filename = f"{idx:03d}_{sanitize_filename(url)}_{timestamp}.png"
                        filepath = os.path.join(config['output_folder'], filename)
                        
                        # Take screenshot
                        await page.screenshot(path=filepath, full_page=config['full_page'])
                        
                        # Verify file was created
                        if os.path.exists(filepath):
                            file_size = os.path.getsize(filepath)
                            screenshots_taken.append(filename)
                            print(f"      ✓ Saved: {filename} ({file_size:,} bytes)")
                        else:
                            print(f"      ❌ Failed to create file: {filename}")
                            
                    except Exception as e:
                        print(f"      ⚠️  Skipped - Error: {str(e)[:100]}")
                    
                    if idx < len(pages_to_capture):
                        await asyncio.sleep(config['wait_between_pages'])
            
        except Exception as e:
            print(f"\n❌ Error occurred: {e}")
            print("\n💡 Troubleshooting:")
            print("  1. Check debug_before_login.png in screenshots folder")
            print("  2. Website may use dynamic loading - increase wait times")
            print("  3. Check if website blocks automation/requires CAPTCHA")
            print("  4. Verify the website URL is correct")
            raise
        
        finally:
            # Close browser
            await browser.close()
            print("\n🔒 Browser closed")
    
    if not screenshots:
        print("\n⚠️  WARNING: No screenshots to add to ZIP!")
        print("   Check if screenshots were successfully saved.")
        return None, 0
    
    zip_path = config['zip_filename']
    added_count = 0
    skipped_count = 0
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for screenshot in screenshots:
            filepath = os.path.join(config['output_folder'], screenshot)
            
            # Check if file exists before adding
            if os.path.exists(filepath):
                try:
                    zipf.write(filepath, screenshot)
                    added_count += 1
                    if added_count <= 10:  # Show first 10
                        print(f"  ✓ Added: {screenshot}")
                    elif added_count == 11:
                        print(f"  ... (showing first 10 only)")
                except Exception as e:
                    print(f"  ❌ Failed to add {screenshot}: {e}")
                    skipped_count += 1
            else:
                print(f"  ⚠️  File not found: {screenshot}")
                skipped_count += 1
    
    if added_count == 0:
        print(f"\n❌ No files were added to ZIP!")
        print(f"   Skipped: {skipped_count} files")
        print(f"   Check if files exist in: {config['output_folder']}/")
        return None, 0
    
    file_size = os.path.getsize(zip_path)
    file_size_mb = file_size / (1024 * 1024)
    
    print(f"\n✅ ZIP created successfully!")
    print(f"  📄 Filename: {zip_path}")
    print(f"  📊 Size: {file_size_mb:.2f} MB ({file_size:,} bytes)")
    print(f"  📸 Screenshots added: {added_count}")
    if skipped_count > 0:
        print(f"  ⚠️  Files skipped: {skipped_count}")
    
    return zip_path, file_size_mb

def create_download_link(zip_path, file_size_mb):
    """Create download link for the ZIP file."""
    print("\n" + "="*70)
    print("🔗 Creating download link...")
    print("="*70 + "\n")
    
    # Check if running in Google Colab
    try:
        from google.colab import files
        print("📍 Google Colab detected - using files.download()\n")
        files.download(zip_path)
        print("✅ Download started! Check your browser downloads.")
        return
    except ImportError:
        pass
    
    # For VS Code or local Jupyter - create base64 link
    print("📍 Local environment detected - creating download link\n")
    
    if file_size_mb > 50:
        print("⚠️  WARNING: File is larger than 50MB")
        print("   Creating download link may take a while...\n")
    
    # Read and encode file
    with open(zip_path, 'rb') as f:
        zip_data = f.read()
    
    b64_data = base64.b64encode(zip_data).decode()
    
    # Create download link
    download_link = f'<a href="data:application/zip;base64,{b64_data}" download="{zip_path}">Click here to download {zip_path}</a>'
    
    html = f"""
    <div style="padding: 20px; border: 2px solid #4CAF50; border-radius: 10px; background-color: #f9f9f9;">
        <h2 style="color: #4CAF50; margin-top: 0;">✅ Screenshots Ready!</h2>
        <p style="font-size: 16px;"><strong>File:</strong> {zip_path}</p>
        <p style="font-size: 16px;"><strong>Size:</strong> {file_size_mb:.2f} MB</p>
        <div style="margin-top: 20px;">
            {download_link}
        </div>
    </div>
    """
    
    display(HTML(html))
    print("\n✅ Download link created! Click the link above to download.")

# Run the automation
try:
    screenshots = await take_screenshots_async(CONFIG)
    
    print(f"\n📋 Summary: {len(screenshots)} screenshot(s) captured")
    
    if len(screenshots) == 0:
        print("\n⚠️  No screenshots were taken!")
        print("   Please check the error messages above.")
    else:
        zip_path, file_size_mb = create_zip(CONFIG, screenshots)
        
        if zip_path:
            create_download_link(zip_path, file_size_mb)
            
            print("\n" + "="*70)
            print("🎉 ALL DONE!")
            print("="*70)
            print(f"\n📦 {len(screenshots)} screenshot(s) packaged successfully!")
            print(f"📁 Location: {CONFIG['output_folder']}/")
            print(f"📥 ZIP File: {zip_path}")
            print("\n💡 TIP: You can edit CONFIG above to add more pages to screenshot!")
        else:
            print("\n❌ Failed to create ZIP file!")
            print(f"   Check files in: {CONFIG['output_folder']}/")
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    print("\n🔍 Full error details:")
    traceback.print_exc()

## 💡 How to Add More Pages to Screenshot

Edit the `CONFIG` in **Step 3** and add URLs to the `pages_to_screenshot` list:

```python
"pages_to_screenshot": [
    "https://rms-ui-dev-slp.supremelifeplatform.com/#/admin/users",
    "https://rms-ui-dev-slp.supremelifeplatform.com/#/business",
    "https://rms-ui-dev-slp.supremelifeplatform.com/#/onboarding-sla",
],
```

Then just re-run **Step 4**!

## 🔧 Troubleshooting

### Login not working?
1. Check if selectors in CONFIG are correct
2. Inspect the login page and update selectors
3. Try setting `headless: False` to see what's happening

### Screenshots incomplete?
1. Increase `wait_before_screenshot` in CONFIG
2. Increase `wait_after_login` for slow-loading pages

### File too large?
- ZIP files >50MB may take time to encode for download
- Consider taking fewer screenshots or lower resolution